<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m5_02_regression_trees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M5.02 — How much alcohol?

In the lab we used a decision tree for **regression**: instead of a
class, each leaf predicts a number, and the prediction looks like a
staircase. In this workbook you predict **the alcohol content of white
wines**. You will count the steps of a tree before drawing it, send a
tree and a line outside the data they know, and see how much a tree
changes when its training data changes a little.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "best_with_two_features": "dc9c5edb8b2d479e",
    "changes_most": "dc9c5edb8b2d479e",
    "impossible_prediction": "9f6183312a353732",
    "leaf_prediction": "4a44dc15364204a8",
    "max_values_depth_5": "e29c9c180c6279b0",
    "n_nodes_depth_2": "7902699be42c8a8e"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Data and target

We keep the white wines of workbook M5.01 and predict `alcohol` (percent
by volume). Measuring alcohol properly needs a distillation; measuring
**density** only needs a scale. Alcohol is lighter than water, so a wine
with more alcohol is less dense. Sugar is heavier than water, so a sweet
wine is denser.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

wine = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/wine_quality.csv"
)
white = wine[wine["color"] == "white"].reset_index(drop=True)
target = white["alcohol"]
print(f"{len(white)} white wines, alcohol from {target.min()} to {target.max()} %")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(white["density"], target, s=4, alpha=0.3)
_ = ax.set(xlabel="density (g/cm³)", ylabel="alcohol (%)", xlim=(0.986, 1.004))

## 2. A staircase

A regression tree asks the same kind of questions as a classification
tree ("is the density below 0.992?"). In each leaf it predicts **the
mean alcohol of the training wines that ended up there**.

🔮 **Predict first.** A tree with `max_depth=2` is trained on the
density alone. How many **different values** can it predict at most?

In [ ]:
my_prediction = ____
commit(my_prediction, list(range(1, 20)))

In [ ]:
from sklearn.tree import DecisionTreeRegressor

density = white[["density"]]
density_axis = pd.DataFrame({"density": np.linspace(0.986, 1.004, 500)})
trees = {depth: DecisionTreeRegressor(max_depth=depth, random_state=0).fit(density, target) for depth in [2, 5]}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, (depth, tree) in zip(axes, trees.items()):
    ax.scatter(white["density"], target, s=4, alpha=0.2)
    ax.plot(density_axis["density"], tree.predict(density_axis), color="black", linewidth=2)
    ax.set(title=f"max_depth={depth}: {tree.get_n_leaves()} leaves", xlabel="density (g/cm³)", xlim=(0.986, 1.004))
    ax.xaxis.set_major_locator(plt.MaxNLocator(5))
_ = axes[0].set_ylabel("alcohol (%)")

✏️ How many different values can a tree with `max_depth=5` predict at
most? (One per leaf; each question doubles the number of groups.)

In [ ]:
max_values_depth_5 = ____
check("max_values_depth_5", max_values_depth_5)

The prediction of a regression tree is always a staircase: flat inside a
leaf, with a jump at each threshold. More depth means more and narrower
steps, and on the right panel some steps already follow a handful of
wines.

✏️ A leaf contains three training wines with 9, 10 and 11 % of alcohol.
What does the tree predict for a new wine that falls in this leaf?

In [ ]:
leaf_prediction = ____
check("leaf_prediction", leaf_prediction, hint="A leaf predicts the mean of its training wines.")

### ➕ Going further: counting the boxes

The tree with `max_depth=2` has 4 leaves. The leaves are only the
**last** boxes: the boxes that ask the questions are nodes of the tree
too.

✏️ How many boxes (nodes) does this tree have **in total**?

In [ ]:
n_nodes_depth_2 = ____
check("n_nodes_depth_2", n_nodes_depth_2, hint="1 box asks the first question, 2 ask the second, and then come the leaves.")

scikit-learn stores a fitted tree as plain arrays, one entry per node:
the feature it asks about, the threshold, and the value it would
predict. `decision_path` gives the nodes a given wine goes through.

In [ ]:
small_tree = trees[2].tree_
nodes = pd.DataFrame(
    {
        "question": [
            f"density <= {small_tree.threshold[node]:.4f}?" if small_tree.children_left[node] != -1 else "(leaf)"
            for node in range(small_tree.node_count)
        ],
        "if yes go to": small_tree.children_left,
        "if no go to": small_tree.children_right,
        "training wines": small_tree.n_node_samples,
        "mean alcohol": small_tree.value.ravel().round(2),
    }
).rename_axis("node")
one_wine = density.iloc[[0]]
print(f"a wine of density {one_wine.iloc[0, 0]} goes through nodes {[int(node) for node in trees[2].decision_path(one_wine).indices]}")
nodes

scikit-learn's example
[Understanding the decision tree structure](https://scikit-learn.org/1.8/auto_examples/tree/plot_unveil_tree_structure.html)
goes further into these arrays.

## 3. Tree or straight line?

The cloud of points in section 1 looks like a line going down. Is a
tree better than a linear regression here? We compare them with the
density alone, and then with the density **and** the residual sugar.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score

cv = KFold(n_splits=5, shuffle=True, random_state=0)
candidates = {
    "linear regression": LinearRegression(),
    "tree, max_depth=5": DecisionTreeRegressor(max_depth=5, random_state=0),
    "tree, no limit": DecisionTreeRegressor(random_state=0),
}
feature_sets = {"density": ["density"], "density + residual_sugar": ["density", "residual_sugar"]}
errors = pd.DataFrame(
    {
        label: {
            name: -cross_val_score(model, white[columns], target, cv=cv, scoring="neg_mean_absolute_error").mean()
            for name, model in candidates.items()
        }
        for label, columns in feature_sets.items()
    }
)
errors.round(3)

The errors are mean absolute errors in percentage points of alcohol.

✏️ With **both** features, which kind of model has the lowest error?
Write `"linear regression"` or `"tree"`.

In [ ]:
best_with_two_features = ____
check("best_with_two_features", best_with_two_features)

🧠 **Think.** With the density alone, the unlimited tree is no better
than the tree limited to depth 5. With two features it is clearly the
best. A wine can be dense because it has little alcohol **or** because
it has a lot of sugar. Why does the second feature help a deep tree so
much?

## 4. ➕ Going further: outside the data

Almost all white wines have a density between 0.987 and 1.003. One wine
of the dataset is far outside: a very sweet wine with a density of
1.039.

In [ ]:
white.sort_values("density", ascending=False).head(3)[["density", "residual_sugar", "alcohol"]]

We train a linear regression and a tree on the density of **all the
other wines**, and ask both what they predict for a density of 1.039.

🔮 **Predict first.** One of the two models will predict an **impossible**
alcohol content. Which one? `"linear regression"` or `"tree"`?

In [ ]:
my_prediction = ____
commit(my_prediction, ["linear regression", "tree"])

In [ ]:
ordinary = white[white["density"] < 1.01]
line = LinearRegression().fit(ordinary[["density"]], ordinary["alcohol"])
tree = DecisionTreeRegressor(max_depth=5, random_state=0).fit(ordinary[["density"]], ordinary["alcohol"])

wide_axis = pd.DataFrame({"density": np.linspace(0.986, 1.040, 500)})
sweet_wine = pd.DataFrame({"density": [1.039]})
outside = {"linear regression": line.predict(sweet_wine)[0], "tree": tree.predict(sweet_wine)[0]}

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.scatter(white["density"], target, s=4, alpha=0.2)
ax.plot(wide_axis["density"], line.predict(wide_axis), label="linear regression")
ax.plot(wide_axis["density"], tree.predict(wide_axis), label="tree", color="black")
ax.axhline(0, color="red", linestyle="--")
ax.set(xlabel="density (g/cm³)", ylabel="alcohol (%)")
ax.legend()
pd.Series(outside, name="predicted alcohol at density 1.039").round(1).to_frame()

✏️ Which model predicts an impossible value for this wine?

In [ ]:
impossible_prediction = ____
check("impossible_prediction", impossible_prediction)

A line keeps going down forever: beyond the data it promises wines with
a **negative** amount of alcohol. A tree predicts the mean of a leaf, so
it can never leave the range of alcohol contents it has seen: beyond the
last threshold its prediction is **flat**.

Flat is not a reasoned answer either. This wine really has 11.7 % of
alcohol. The tree happens to land near it, but only by repeating the
mean of its last leaf: it would give the same answer for a density of
1.1 or 2. Neither model knows the real reason for this density (a huge
amount of sugar), because it is not in the feature they were given.

In the lab, the flat prediction of trees was presented as a weakness
(they cannot follow a trend). Here it protects against an absurd answer.
It is the same property: **a tree cannot extrapolate**.

🧠 **Think.** In workbook M4.02 a polynomial predicted absurd traffic
for the following months. Would a tree have predicted something absurd?
Would it have predicted a rising trend?

## 5. ➕ Going further: a small change in the data

We draw two random samples of 300 wines, as if two laboratories had each
analysed their own bottles (the "laboratories" of workbook M2.03). Each
trains an unlimited tree and a linear regression.

🔮 **Predict first.** For which kind of model will the two laboratories
end up with the most different predictions? `"linear regression"` or
`"tree"`?

In [ ]:
my_prediction = ____
commit(my_prediction, ["linear regression", "tree"])

In [ ]:
samples = [ordinary.sample(n=300, random_state=seed) for seed in (1, 2)]
kinds = {"linear regression": LinearRegression(), "tree": DecisionTreeRegressor(random_state=0)}

disagreement = {}
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, (name, model) in zip(axes, kinds.items()):
    curves = [model.fit(sample[["density"]], sample["alcohol"]).predict(density_axis) for sample in samples]
    for curve, label in zip(curves, ["laboratory 1", "laboratory 2"]):
        ax.plot(density_axis["density"], curve, label=label, linewidth=1)
    disagreement[name] = np.abs(curves[0] - curves[1]).mean()
    ax.set(title=f"{name}: mean disagreement {disagreement[name]:.2f}", xlabel="density (g/cm³)")
    ax.xaxis.set_major_locator(plt.MaxNLocator(5))
axes[0].set_ylabel("predicted alcohol (%)")
_ = axes[0].legend()

✏️ Which kind of model changes the most when the training data changes?

In [ ]:
changes_most = ____
check("changes_most", changes_most)

The two lines can hardly be told apart. The two trees agree on the
general direction and disagree almost everywhere on the details: each
has built its steps around its own 300 wines. In the words of module 2,
a deep tree is a **high-variance** model.

This weakness is the starting point of the next module: if one tree is
unstable, train many of them and average.

🧠 **Think.** Which hyperparameter of the tree would you change to make
the two laboratories agree more? What would you lose?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - why the prediction of a regression tree is a staircase, and how
     many steps and nodes a tree of a given depth can have;
   - tree against linear regression with one and with two features;
   - what the line and the tree predict for the very dense wine, and
     why neither can be trusted there;
   - the two laboratories: which model is stable, and what that
     announces for the next module.